In [ ]:
import json
from pathlib import Path

import numpy as np
from PIL import Image

root = Path("../data")
index = json.loads((root / "index.json").read_text())
names = [i["name"] for i in index if i["split"] == "train"]

name = names[0]
rgb = np.asarray(Image.open(root / "frames" / f"{name}_rgb.png").convert("RGB"))
depth = np.load(root / "frames" / f"{name}_depth.npy")
meta = json.loads((root / "frames" / f"{name}.json").read_text())

points = np.load(root / "frames" / f"{name}_lidar.npy")
column, row, z = points[:, 0].astype(int), points[:, 1].astype(int), points[:, 2]
lidar = np.zeros_like(depth)
lidar[row, column] = z


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

CORRECTION_LIMIT = 3.0
PIN_PIXELS = 6.0


def block(ins, outs):
    return nn.Sequential(
        nn.Conv2d(ins, outs, 3, padding=1),
        nn.GroupNorm(8, outs),
        nn.ReLU(),
        nn.Conv2d(outs, outs, 3, padding=1),
        nn.GroupNorm(8, outs),
        nn.ReLU(),
    )


class SnowRefiner(nn.Module):
    def __init__(self, use_lidar=True, sensor_dropout=0.3):
        super().__init__()
        self.use_lidar = use_lidar
        self.sensor_dropout = sensor_dropout

        # base 1 + lidar 3 + motion 3 + rgb 3 = 10 channels. Motion is off for now, so its 3 channels stay empty
        self.enc1 = block(10, 32)
        self.enc2 = block(32, 64)
        self.enc3 = block(64, 128)
        self.enc4 = block(128, 256)
        self.middle = block(256, 256)

        layer = nn.TransformerEncoderLayer(256, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=4)
        self.position = nn.Parameter(torch.zeros(1, 256, 45, 80))

        self.scale = nn.Linear(256, 1)
        self.dec4 = block(256 + 256, 256)
        self.dec3 = block(256 + 128, 128)
        self.dec2 = block(128 + 64, 64)
        self.dec1 = block(64 + 32, 32)
        self.out = nn.Conv2d(32, 2, 1)

        # starts at zero, so the untrained model returns the DA3 depth
        for layer in (self.scale, self.out):
            nn.init.zeros_(layer.weight)
            nn.init.zeros_(layer.bias)

    def sensor_on(self, x):
        # 1 keeps LiDAR and 0 switches it off, chosen per sample while training
        on = torch.full((len(x), 1, 1, 1), float(self.use_lidar), device=x.device)
        if self.training:
            on = on * (torch.rand_like(on) >= self.sensor_dropout)
        return on

    def up(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode="bilinear")
        return torch.cat([x, skip], dim=1)

    def forward(self, rgb, base, filled, distance):
        on = self.sensor_on(rgb)
        log_base = base.clamp(min=1e-3).log()
        has = (filled > 0).float()
        log_filled = filled.clamp(min=1e-3).log()

        # LiDAR goes in as its depth, how close the nearest point is, and how wrong DA3 is there
        near = torch.exp(-distance / 16) * has
        gap = (log_filled - log_base) * has
        empty = torch.zeros_like(base)
        x = torch.cat([log_base, filled.log1p() * on, near * on, gap * on, empty, empty, empty, rgb], dim=1)

        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e3 = self.enc3(F.max_pool2d(e2, 2))
        e4 = self.enc4(F.max_pool2d(e3, 2))
        m = self.middle(F.max_pool2d(e4, 2))

        # residual skip: what goes into the transformer also goes straight past it
        b, c, h, w = m.shape
        tokens = m + F.interpolate(self.position, size=(h, w), mode="bilinear")
        m = m + self.transformer(tokens.flatten(2).transpose(1, 2)).transpose(1, 2).reshape(b, c, h, w)

        # one scale for the whole frame first
        frame_scale = self.scale(m.mean(dim=(2, 3)))[:, :, None, None]

        d = self.dec4(self.up(m, e4))
        d = self.dec3(self.up(d, e3))
        d = self.dec2(self.up(d, e2))
        d = self.dec1(self.up(d, e1))

        # then a capped correction for each pixel
        correction, uncertainty = self.out(d).chunk(2, dim=1)
        free = log_base + frame_scale + CORRECTION_LIMIT * torch.tanh(correction)

        # right next to a real point, trust the point. Further away, the network takes over
        pin = torch.exp(-distance / PIN_PIXELS) * has * on
        return (1 - pin) * free + pin * log_filled, uncertainty

In [ ]:
BANDS = [0, 10, 30, 100, 300, 1000, 3000, 1e9]


def depth_loss(log_depth, uncertainty, exact):
    # sky is 0 in the exact depth, so it's left out
    valid = exact > 0
    e, p, s = exact[valid], log_depth[valid].float(), uncertainty[valid].float()

    # the paper's loss, plus real AbsRel, which punishes overshooting on nearby pixels
    pixel = torch.abs(p - e.log()) * torch.exp(-s) + s + torch.abs(p.exp() - e) / e

    # each distance band counts the same, so far terrain can't drown out close range
    total = 0
    used = 0
    for low, high in zip(BANDS, BANDS[1:]):
        part = (e >= low) & (e < high)
        if part.any():
            total = total + pixel[part].mean()
            used += 1
    return total / used

In [ ]:
from torch.utils.data import DataLoader

# the dataset is in its own file because workers can't load a class from a notebook on Windows
from snow_data import SnowDataset

# fastest settings that fit in the GPU: batch 4 runs out of real memory and gets about 16x slower
train_loader = DataLoader(SnowDataset("train", fresh=True), batch_size=2, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(SnowDataset("val"), batch_size=1, num_workers=4, pin_memory=True, persistent_workers=True)

In [ ]:
device = "cuda"
EPOCHS = 26
model = SnowRefiner().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))
scaler = torch.amp.GradScaler()


def validate():
    model.eval()
    absrel = 0
    log_error = 0
    with torch.no_grad():
        for batch in val_loader:
            rgb, base, filled, distance, exact = [x.to(device) for x in batch]

            with torch.amp.autocast(device):
                log_depth, uncertainty = model(rgb, base, filled, distance)

            valid = exact > 0
            predicted = log_depth.float().exp()[valid]
            absrel += (torch.abs(predicted - exact[valid]) / exact[valid]).mean().item()
            log_error += torch.abs(predicted.log() - exact[valid].log()).mean().item()
    return absrel / len(val_loader), log_error / len(val_loader)


best = float("inf")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for batch in train_loader:
        rgb, base, filled, distance, exact = [x.to(device) for x in batch]

        with torch.amp.autocast(device):
            log_depth, uncertainty = model(rgb, base, filled, distance)
            loss = depth_loss(log_depth, uncertainty, exact)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        schedule.step()
        train_loss += loss.item()

    absrel, log_error = validate()
    print(f"epoch {epoch + 1}: train loss {train_loss / len(train_loader):.3f}, val AbsRel {absrel:.3f}, log error {log_error:.3f}", flush=True)

    # refiner_v5.pt holds the reported results, so a new run saves under its own name
    torch.save(model.state_dict(), "refiner_last.pt")
    if absrel < best:
        best = absrel
        torch.save(model.state_dict(), "refiner_best.pt")